In [1]:
import cv2
import pandas as pd
import seaborn as sb
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

In [2]:
df = pd.read_csv("bitcoin.csv")

In [ ]:
df.head(-5)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
plt.close()
plt.figure(figsize=(12,5))
plt.plot(df['Close'])
plt.title('Bitcoin Close price.',fontsize=15)
plt.ylabel('Price in dollers')
plt.show()

In [ ]:
df.isnull().sum()

In [4]:
spllited = df['Date'].str.split('-',expand=True)

In [5]:
df['year'] = spllited[0].astype('int')
df['month'] = spllited[1].astype('int')
df['day'] = spllited[2].astype('int')

In [ ]:
df.head()

In [ ]:
plt.close()
data_grouped = df.groupby('year')[['Open','High','Low','Close']].mean()
plt.figure(figsize = (12,7))
for i,col in enumerate(['Open','High','Low','Close']):
    plt.subplot(2,2,i+1)
    data_grouped[col].plot.bar()
plt.show()

In [7]:
df['is_quarter_end'] = np.where(df['month'] % 3 == 0, 1 , 0)

In [8]:
df['Open-Close'] = df['Open'] - df['Close']
df['Low-High'] = df['Low'] - df['High']

In [9]:
df['target'] = np.where(df['Close'].shift(-1) > df['Close'] , 1 , 0)

In [ ]:
plt.close()
plt.pie(df['target'].value_counts().values,labels=[0,1],autopct='%1.1f%%')
plt.show()

In [ ]:
df.head()

In [ ]:
feachers = df[['Open-Close' , 'Low-High' , 'is_quarter_end']]
target = df[['target']]
scaler = StandardScaler()
feachers = scaler.fit_transform(feachers)
X_train,X_valid,y_train,y_valid = train_test_split(feachers,target,test_size=0.15,random_state=2020)
print(X_train.shape,X_valid.shape)

In [ ]:
models = [LogisticRegression(),DecisionTreeClassifier(),KNeighborsClassifier()]
for i in range(3):
    models[i].max_iter = 10000
    models[i].fit(X_train,y_train)

    print(f"{models[i]} :")
    print('Traning Accuracy :', metrics.roc_auc_score(y_train,models[i].predict_proba(X_train)[:,1]))
    print('Validation Accuracy :', metrics.roc_auc_score(y_valid,models[i].predict_proba(X_valid)[:,1]))
    print()